In [1]:
#Starter-script for Prophet-konkurranse
#--------------------------------------
#
#- Leser train.csv
#- Leser test_features.csv
#- Trener Prophet-modell
#- Lager submission.csv

#Du kan forbedre:
#- features
#- skalering
#- lag-features
#- hyperparametre
#- log-transform
#- osv.

In [2]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.preprocessing import StandardScaler

In [3]:
lagnsavn="feels_like_sept_2022"

In [4]:
# --------------------------------------------------
# 1. LES DATA
# --------------------------------------------------
print("Leser data ...")
train = pd.read_csv("https://raw.githubusercontent.com/jensmorten/sykkelprofet/refs/heads/main/konkurranse/bysykkel_train.csv", parse_dates=["ds"])
test = pd.read_csv("https://raw.githubusercontent.com/jensmorten/sykkelprofet/refs/heads/main/konkurranse/test_compete.csv", parse_dates=["ds"])
train = train.sort_values("ds")
test = test.sort_values("ds")
print("done!")

Leser data ...
done!


In [5]:
train=train[train['year']>=2022]

In [6]:
test=test[test['year']>=2022]

In [7]:
weather_cols = [
    "air_temperature",
    "wind_speed",
    "precipitation_amount"
]

train[weather_cols] = (
    train[weather_cols]
    .ffill()
    .bfill()
)

In [8]:
def compute_feels_like(temp, wind_speed):
    """
    Wind chill formula (Environment Canada) for temp < 10°C.
    For temp >= 10°C, returns the raw temperature.
    Wind speed should be in m/s; formula expects km/h.
    """
    wind_kmh = wind_speed * 3.6  # Convert m/s to km/h
    wind_chill = (
        13.12
        + 0.6215 * temp
        - 11.37 * np.power(np.maximum(wind_kmh, 1), 0.16)
        + 0.3965 * temp * np.power(np.maximum(wind_kmh, 1), 0.16)
    )
    return np.where(temp < 10, wind_chill, temp)

In [9]:
train['sept_ind'] = (train['month']==9).astype(int)
test['sept_ind']  = (test['month']==9).astype(int)

In [10]:
train["feels_like"] = compute_feels_like(
    train["air_temperature"].values,
    train["wind_speed"].values
)
test["feels_like"] = compute_feels_like(
    test["air_temperature"].values,
    test["wind_speed"].values
)

In [11]:
regressors = [ 
    "precipitation_amount",
    "feels_like",
    "sept_ind"
]

In [12]:
train["y_original"] = train["y"].copy()
train["y"] = np.log1p(train["y"])  # log(1 + y) to handle y=0

print(f"Original y range: {train['y_original'].min():.0f} – {train['y_original'].max():.0f}")
print(f"Log y range: {train['y'].min():.2f} – {train['y'].max():.2f}")

Original y range: 1 – 540
Log y range: 0.69 – 6.29


In [13]:
# --------------------------------------------------
# 4. DEFINER PROPHET
# --------------------------------------------------

m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=True,
    seasonality_mode="additive"
)

m.add_country_holidays(country_name='NO')

for r in regressors:
    m.add_regressor(r)

In [14]:
print("Trener modell ...")
m.fit(train[["ds", "y"] + regressors])

Trener modell ...


09:43:19 - cmdstanpy - INFO - Chain [1] start processing
09:43:20 - cmdstanpy - INFO - Chain [1] done processing


In [15]:
# --------------------------------------------------
# 6. PREDIKSJON PÅ TEST
# --------------------------------------------------
future = test[["ds"] + regressors].copy()
forecast = m.predict(future)

In [16]:
submission = pd.DataFrame({
    "ds": test["ds"],
    "yhat": forecast["yhat"]
})

submission["yhat"] = np.expm1(submission["yhat"])  # exp(yhat) - 1


In [17]:
#lagre innlevering
submission.to_csv(f"submission_{lagnsavn}.csv", index=False, sep=',')
print("✅ Ferdig!")
print("Submission lagret")

✅ Ferdig!
Submission lagret
